# Complexity Analysis — How to Derive It

*Method — read once, then apply. For looking up the cost of a built-in operation, use `04_complexity_reference` instead.*

Interviewers do not ask you to recall a complexity; they ask you to derive one for code you have just written, often code that is not in any table. This notebook covers the four techniques that account for almost every derivation you will need: **loop counting**, **recursion trees**, **amortised reasoning**, and **space including the call stack**.

Worked applications live in the topic notebooks — the amortised argument for the monotonic stack is in `09_heap_and_monotonic_stack`, the recursion-tree counts for subsets and permutations are in `12_backtracking`.

## Quick Index

| Section | Content |
| :--- | :--- |
| §1 | Loop counting |
| §2 | Recursion trees |
| §3 | Amortised analysis |
| §4 | Space complexity, including the recursion stack |
| §5 | Complexities worth recognising on sight |
| §6 | Common mistakes |

---
## §1 — Loop counting

Two rules cover most cases: **nested loops multiply, sequential loops add.**

Sequential blocks add and the larger dominates, so an O(n) pass followed by an O(n log n) sort is O(n log n) overall. Nested loops multiply — but only when the inner bound is independent of the outer one.

**The case people get wrong** is when the inner bound depends on the outer variable:

```python
for i in range(n):
    for j in range(i, n):    # NOT n iterations — it is n - i
        ...
```

This is not O(n²) by the multiplication rule; you have to sum the series. The inner loop runs `n − i` times, so the total is `n + (n−1) + ... + 1 = n(n+1)/2`, which is **O(n²)** anyway — but only because the series happens to sum that way. The same shape with a *halving* inner bound is very different:

```python
i = n
while i > 0:
    for j in range(i):       # i, then i/2, then i/4, ...
        ...
    i //= 2
```

Here the total is `n + n/2 + n/4 + ... = 2n`, which is **O(n)**, not O(n log n). A geometric series is dominated by its largest term. This is the single most useful fact in loop counting, and it is also the engine behind amortised analysis in §3.

**Loops that multiply by a non-`n` factor.** When the inner loop is bounded by something other than the input size — the alphabet, the number of bits, a fixed k — carry it in the complexity rather than dropping it. `O(n · 26)` is honest and simplifies to O(n); `O(n · m)` where `m` is another input does not simplify at all, and interviewers notice when you drop it.

---
## §2 — Recursion trees

For a recursive function, draw the call tree and count the nodes. Two numbers determine everything:

- **b** — the branching factor: how many recursive calls each invocation makes.
- **d** — the depth: how many levels before you hit the base case.

The tree has roughly **b^d** nodes. Multiply by the work done *at* each node (excluding the recursive calls) to get the total.

| Problem | Branching | Depth | Work per node | Total |
| :--- | :--- | :--- | :--- | :--- |
| Subsets — include or exclude each element | 2 | n | O(1) | **O(2ⁿ)** |
| Subsets, copying each result | 2 | n | O(n) to copy | **O(n · 2ⁿ)** |
| Permutations — choose from remaining | n, then n−1, then n−2 … | n | O(1) | **O(n!)** |
| Permutations, copying each result | as above | n | O(n) | **O(n · n!)** |
| Naive Fibonacci | 2 | n | O(1) | **O(2ⁿ)** |
| Memoised Fibonacci | 2, but each state computed once | — | O(1) | **O(n)** |
| Binary search | 1 | log n | O(1) | **O(log n)** |
| Merge sort | 2 | log n | O(n) merge per *level* | **O(n log n)** |

**Why permutations is n! and not nⁿ.** The branching factor shrinks: the root has n choices, each child has n−1 remaining, and so on. The product `n × (n−1) × ... × 1` is n!, not nⁿ. Subsets keeps a constant branching factor of 2 at every level, which is why it is 2ⁿ.

**Why merge sort is n log n.** Do not count nodes here — count *levels*. There are log n levels, and every level does O(n) total merging work regardless of how many nodes it is split across. Levels × work-per-level is the right frame whenever the work per node shrinks in proportion to the branching.

**Memoisation changes the count entirely.** Once results are cached, the tree collapses: the total is **(number of distinct states) × (work per state)**, and the branching factor becomes irrelevant. This is the whole reason DP is faster than backtracking on the same recurrence. For a DP with a 2D state over `i` and `j` and O(1) transitions, that is O(n · m) — count the states, not the calls.

---
## §3 — Amortised analysis

Some loops look quadratic and are not. When an inner `while` loop is bounded by *how much work earlier iterations created* rather than by n, the correct total is the sum over the whole run, not the worst case of a single iteration multiplied by n.

**The accounting argument.** Find a quantity that each unit of inner work consumes, bound how much of it can ever be created, and the total inner work is bounded by that.

**Monotonic stack.**

```python
for x in nums:
    while stack and stack[-1] < x:   # looks like it could run n times
        stack.pop()
    stack.append(x)
```

The inner `while` can indeed run n times on a single iteration. But each iteration pushes exactly **one** element, so across the whole loop at most n elements are ever pushed — and each can be popped at most once. Total pops ≤ n, so the whole thing is **O(n)**, not O(n²). The unit being consumed is "an element on the stack", and only n are ever created.

**Sliding window.**

```python
for right in range(n):
    while invalid(window):
        left += 1        # never moves backwards
```

`left` only ever increases and is bounded by n, so the inner loop executes at most n times *in total* across all iterations of the outer loop. Outer O(n) plus inner O(n) total is **O(n)**. The unit here is "positions the left pointer can advance", of which there are n.

**`list.append`.** Appending is O(1) amortised, not O(1) worst case: when the backing array is full, CPython allocates a larger one and copies everything, which is O(n) for that single call. But resizes are geometric, so across n appends the total copying is `1 + 2 + 4 + ... + n < 2n` — the same geometric-series argument from §1. Amortised O(1).

**The test for whether amortised reasoning applies:** ask whether the inner loop's work is bounded by something *created* by earlier iterations and *consumed* permanently. If work can be redone — as in a nested scan that restarts from zero each time — it is genuinely quadratic and amortised reasoning does not apply.

---
## §4 — Space complexity

Space is the more commonly botched half, because the parts that count are easy to forget.

**Count all four of these:**

1. Explicit data structures you allocate.
2. **The recursion call stack** — depth × frame size.
3. Copies made by slicing, `sorted()`, or building result lists.
4. Space used by the output, *if* the question counts it.

**The recursion stack is the one people miss.** This DFS is routinely described as O(1) space:

```python
def maxDepth(root):
    if not root: return 0
    return 1 + max(maxDepth(root.left), maxDepth(root.right))
```

It allocates nothing, but at its deepest point there are **h** stack frames live, where `h` is the height of the tree. So the space is **O(h)** — O(log n) for a balanced tree, and **O(n)** for a degenerate one shaped like a linked list. Always state it as O(h) and then give the two bounds; saying "O(1)" is wrong and saying "O(log n)" is only true for balanced input.

The same applies elsewhere:

- **Backtracking:** O(depth) for the stack plus O(depth) for the current partial candidate — usually O(n) — *excluding* the output, which can be exponential.
- **Merge sort:** O(n) for the merge buffer plus O(log n) for the stack, so O(n).
- **Quicksort:** O(log n) expected stack depth, O(n) worst case if the pivots are bad.
- **Iterative BFS:** no recursion stack, but the queue can hold an entire level — O(w) where `w` is the maximum width, which is O(n) for the last level of a complete tree.

**Output space convention.** For a problem that must return every subset, the output is O(2ⁿ) and there is nothing to be done about it. The convention is to state auxiliary space separately: "O(n) auxiliary, excluding the O(n · 2ⁿ) output". Say which convention you are using rather than leaving it ambiguous.

**Sneaky allocations.** `s[i:j]` copies. `sorted(x)` builds a new list while `x.sort()` does not. `list(range(n))` materialises n integers while `range(n)` does not. Passing a slice into a recursive call — `helper(nums[1:])` — turns an O(n) recursion into O(n²) time *and* space, which is why the index-passing form `helper(nums, i + 1)` is standard.

---
## §5 — Complexities worth recognising on sight

| Shape you see | Complexity |
| :--- | :--- |
| Halving the search space each step | O(log n) |
| One pass, O(1) work per element | O(n) |
| One pass with an amortised inner `while` | O(n) |
| Sort, then a single pass | O(n log n) |
| Heap of size k, pushed n times | O(n log k) |
| Every pair | O(n²) |
| 2D DP over an n × m table, O(1) transitions | O(n · m) |
| 2D DP with an O(n) transition inside each cell | O(n³) |
| Include-or-exclude over n items | O(2ⁿ) |
| All orderings of n items | O(n!) |
| Bitmask DP over subsets with an O(n) transition | O(n · 2ⁿ) |

**A quick sanity check against the constraints:** roughly 10⁸ elementary operations per second is the working assumption. If `n = 10⁵` and your solution is O(n²), that is 10¹⁰ — three orders of magnitude too slow, so stop coding and rethink. Table 1 in `01_decision_guide` inverts this: it starts from the constraint and tells you which complexities are admissible.

---
## §6 — Common mistakes

| Mistake | Symptom | Correction |
| :--- | :--- | :--- |
| Calling a DFS "O(1) space" | Interviewer pushes back; MLE on a skewed tree | It is O(h) — the recursion stack |
| Dropping the copy cost of results | Stated O(2ⁿ) for subsets, actually O(n · 2ⁿ) | Multiply node count by work per node |
| Multiplying instead of summing for nested amortised loops | Stated O(n²) for a monotonic stack | Each element is pushed and popped once → O(n) |
| Treating `bisect.insort` as O(log n) | TLE despite a "log n" analysis | The search is O(log n); the shift is O(n) |
| Ignoring the length of hashed keys | Stated O(n), actually O(n · k) | Hashing a string is proportional to its length |
| Dropping a second input variable | Stated O(n²) when it is O(n · m) | Keep both symbols when both are inputs |
| Counting recursive *calls* under memoisation | Stated O(2ⁿ) for memoised DP | Count distinct *states* × work per state |
| Forgetting slices copy | Stated O(n) for a recursion that passes `nums[1:]` | O(n²); pass an index instead |
| Assuming `heapify` is O(n log n) | Missing an easy optimisation | `heapify` is O(n); repeated push is O(n log n) |
| Quoting average case for a hash-heavy solution without saying so | Fine in practice, but state the assumption | "O(1) average for dict lookups" |